In [0]:

from datetime import datetime

modo = "historico" # "automatico"


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None

  

In [0]:
if modo == "automatico":
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW tvw_fact_rentabilidad_cliente AS

        WITH ingresos_mov AS (
            SELECT
                id_cli
                ,periodo
                ,SUM(CASE WHEN UPPER(tip_mov) = 'INTERES'  THEN vr_mov ELSE 0 END) AS INGRESOS_INTERESES
                ,SUM(CASE WHEN UPPER(tip_mov) = 'COMISION'  THEN vr_mov ELSE 0 END) AS INGRESOS_COMISIONES_MOV
            FROM silver.cleaned.tb_mov_financieros
            WHERE UPPER(cod_estado_mov) = 'EXITOSO'
            GROUP BY id_cli, periodo
        ),

        ingresos_com AS (
            SELECT
                id_cli
                ,periodo
                ,SUM(vr_comision)                                                    AS INGRESOS_COMISIONES_LOG
            FROM silver.cleaned.tb_comisiones_log
            WHERE UPPER(estado_cobro) = 'COBRADO'
            GROUP BY id_cli, periodo
        ),

        base AS (
            SELECT
                COALESCE(m.id_cli,  c.id_cli)          AS ID_CLIENTE
                ,COALESCE(m.periodo, c.periodo)          AS PERIODO
                ,COALESCE(m.INGRESOS_INTERESES,      0)  AS INGRESOS_INTERESES
                ,COALESCE(m.INGRESOS_COMISIONES_MOV, 0)  AS INGRESOS_COMISIONES_MOV
                ,COALESCE(c.INGRESOS_COMISIONES_LOG, 0)  AS INGRESOS_COMISIONES_LOG
            FROM ingresos_mov m
            FULL OUTER JOIN ingresos_com c
                ON  m.id_cli  = c.id_cli
                AND m.periodo = c.periodo
        ),

        base_enriquecida AS (
            SELECT
                b.ID_CLIENTE
                ,b.PERIODO
                ,b.INGRESOS_INTERESES
                ,b.INGRESOS_COMISIONES_MOV
                ,b.INGRESOS_COMISIONES_LOG
                ,b.INGRESOS_INTERESES
                + b.INGRESOS_COMISIONES_MOV
                + b.INGRESOS_COMISIONES_LOG                              AS INGRESO_TOTAL
            FROM base b
            INNER JOIN gold.financiero.dim_clientes d
                ON b.ID_CLIENTE = d.ID_CLIENTE
        )

        SELECT
            ID_CLIENTE
            ,PERIODO
            ,INGRESOS_INTERESES
            ,INGRESOS_COMISIONES_MOV
            ,INGRESOS_COMISIONES_LOG
            ,INGRESO_TOTAL
            ,ROUND(SUM(INGRESO_TOTAL) OVER (
                PARTITION BY ID_CLIENTE
                ORDER BY PERIODO
                ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
            ), 2)                                                         AS CLTV_12M
            ,CURRENT_DATE                                                 AS _FECHA_CARGA
            ,'tb_mov_financieros+tb_comisiones_log'                       AS _FUENTE

        FROM base_enriquecida
    """)

    spark.sql(f"""
        DELETE FROM gold.financiero.fact_rentabilidad_cliente
        WHERE PERIODO = '{periodo}'
    """)

    spark.sql("""
        INSERT INTO gold.financiero.fact_rentabilidad_cliente
        SELECT
            ID_CLIENTE                   
            ,PERIODO                     
            ,INGRESOS_INTERESES           
            ,INGRESOS_COMISIONES_MOV      
            ,INGRESOS_COMISIONES_LOG     
            ,INGRESO_TOTAL                
            ,CLTV_12M                     
            ,_FECHA_CARGA                 
            ,_FUENTE                 
        FROM tvw_fact_rentabilidad_cliente
    """)

    spark.sql("""
        INSERT INTO gold.financiero.fact_rentabilidad_cliente
        SELECT
            m.ID_MOVIMIENTO                                                  AS ID_MOVIMIENTO
        FROM tvw_fact_rentabilidad_cliente
    """)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.financiero.fact_rentabilidad_cliente (
     ID_CLIENTE                   STRING,
     PERIODO                      STRING,
     INGRESOS_INTERESES           DOUBLE,
     INGRESOS_COMISIONES_MOV      DOUBLE,
     INGRESOS_COMISIONES_LOG      DOUBLE,
     INGRESO_TOTAL                DOUBLE,
     CLTV_12M                     DOUBLE,
     _FECHA_CARGA                 DATE,
     _FUENTE                      STRING
)
USING DELTA
PARTITIONED BY (PERIODO)
LOCATION 'abfss://gold@stdataknowdeveastus001.dfs.core.windows.net/financiero/fact_rentabilidad_cliente';